In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install -q transformers==4.46.3 datasets accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 65.3 MB/s eta 0:00:00


## 1. Config & Seed

In [3]:
import os, gc, random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    set_seed,
)

SEED = 2025
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# ---- Key knobs ----
MODEL_NAME = "microsoft/deberta-v3-base"   # swap to "microsoft/deberta-v3-large" for higher score if GPU time allows
MAX_LEN = 320
N_FOLDS = 5                                 # set to 1 for a quick single-split sanity run
EPOCHS = 6
LR = 1e-5
TRAIN_BS = 4
EVAL_BS = 8
GRAD_ACCUM = 2

OPTIONS = ["A", "B", "C", "D", "E"]


CUDA Available: True
GPU: Tesla T4


## 2. Load data

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print("Train:", train.shape, " Test:", test.shape)
train["label"] = train["answer"].map({o: i for i, o in enumerate(OPTIONS)})
display(train.head())


Train: (2000, 8)  Test: (500, 7)


,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,0


## 3. MAP@3 — implemented exactly as the competition scores it

For each question, given a ranked list of up to 3 predicted labels, the score is
`1` if the true label is 1st, `1/2` if 2nd, `1/3` if 3rd, else `0`. MAP@3 is the mean of that over all questions.
We use this both as the `Trainer`'s `compute_metrics` (so checkpoint selection matches the leaderboard metric)
and as our local cross-validation score.

In [5]:
def map_at_3_from_probs(probs, true_idx):
    """probs: (n_samples, 5) array of option probabilities. true_idx: (n_samples,) int array of correct option index."""
    top3 = np.argsort(-probs, axis=1)[:, :3]
    scores = np.zeros(len(true_idx))
    for rank in range(3):
        hit = (top3[:, rank] == true_idx)
        scores[hit] = 1.0 / (rank + 1)
    return scores.mean()


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)
    acc = accuracy_score(labels, preds)
    map3 = map_at_3_from_probs(probs, np.asarray(labels))
    return {"accuracy": acc, "map3": map3}


## 4. Multiple-choice preprocessing + dynamic-padding collator

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def preprocess_mc(examples, has_labels=True):
    n = len(examples["prompt"])
    first_sentences = [[q] * 5 for q in examples["prompt"]]
    second_sentences = [[examples[opt][i] for opt in OPTIONS] for i in range(n)]

    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])

    tokenized = tokenizer(
        [str(s) for s in first_sentences],
        [str(s) for s in second_sentences],
        truncation=True,
        max_length=MAX_LEN,
    )

    out = {k: [v[i:i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}
    if has_labels:
        out["label"] = examples["label"]
    return out


from dataclasses import dataclass
from typing import Optional, Union
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy


@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0].keys() else "labels"
        has_labels = label_name in features[0]
        labels = [feature.pop(label_name) for feature in features] if has_labels else None

        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        flattened_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features
        ]
        flattened_features = sum(flattened_features, [])

        batch = self.tokenizer.pad(
            flattened_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if has_labels:
            batch["labels"] = torch.tensor(labels, dtype=torch.int64)
        return batch


data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


## 5. Stratified K-Fold training

One row = one question, so this split is leak-free (unlike the baseline's row-level split on the exploded
binary dataframe). We keep out-of-fold (OOF) predictions to compute an honest local MAP@3, and we keep each
fold's test-set probabilities to average into an ensemble.

In [7]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs = np.zeros((len(train), 5))
test_probs_folds = []  # list of (n_test, 5) arrays, one per fold

test_ds_raw = Dataset.from_pandas(test.reset_index(drop=True))
test_ds_tok = test_ds_raw.map(
    lambda ex: preprocess_mc(ex, has_labels=False),
    batched=True,
    remove_columns=test_ds_raw.column_names,
)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, train["label"])):
    print(f"\n===== FOLD {fold + 1}/{N_FOLDS} =====")

    tr_fold = train.iloc[tr_idx].reset_index(drop=True)
    va_fold = train.iloc[va_idx].reset_index(drop=True)

    tr_ds = Dataset.from_pandas(tr_fold)
    va_ds = Dataset.from_pandas(va_fold)

    tr_ds = tr_ds.map(lambda ex: preprocess_mc(ex, True), batched=True, remove_columns=tr_ds.column_names)
    va_ds = va_ds.map(lambda ex: preprocess_mc(ex, True), batched=True, remove_columns=va_ds.column_names)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

    args = TrainingArguments(
        output_dir=f"mc_model_fold{fold}",
        overwrite_output_dir=True,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        weight_decay=0.01,
        warmup_ratio=0.10,
        lr_scheduler_type="cosine",
        per_device_train_batch_size=TRAIN_BS,
        per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=torch.cuda.is_available(),
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="map3",
        greater_is_better=True,
        logging_steps=50,
        save_total_limit=1,
        report_to="none",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=va_ds,
        data_collator=data_collator,
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # ---- OOF predictions for this fold's validation rows ----
    va_pred = trainer.predict(va_ds)
    va_probs = torch.softmax(torch.tensor(va_pred.predictions), dim=1).numpy()
    oof_probs[va_idx] = va_probs

    fold_map3 = map_at_3_from_probs(va_probs, train.iloc[va_idx]["label"].values)
    print(f"Fold {fold + 1} MAP@3 (validation): {fold_map3:.5f}")

    # ---- Test predictions for this fold ----
    test_pred = trainer.predict(test_ds_tok)
    test_probs = torch.softmax(torch.tensor(test_pred.predictions), dim=1).numpy()
    test_probs_folds.append(test_probs)

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

overall_oof_map3 = map_at_3_from_probs(oof_probs, train["label"].values)
print(f"\n================================")
print(f"Overall OOF MAP@3 (this should track your leaderboard score): {overall_oof_map3:.5f}")
print(f"================================")


Map:   0%|          | 0/500 [00:00<?, ? examples/s]


===== FOLD 1/5 =====


Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

...

Some weights of DebertaV2ForMultipleChoice were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_24/1461123279.py:50: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Map3
1,1.583700,1.479263,0.650000,0.781667
2,0.954200,0.891683,0.822500,0.897917
3,0.592700,0.484819,0.890000,0.928750
4,0.402300,0.334175,0.902500,0.941667
5,0.327700,0.285725,0.915000,0.947917
6,0.307600,0.278922,0.917500,0.949167


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Fold 5 MAP@3 (validation): 0.94917



Overall OOF MAP@3 (this should track your leaderboard score): 0.96842


## 6. Ensemble test-set probabilities across folds

In [8]:
test_probs_ensemble = np.mean(test_probs_folds, axis=0)  # (n_test, 5)
np.save("/kaggle/working/test_probs_ensemble.npy", test_probs_ensemble)
np.save("/kaggle/working/oof_probs.npy", oof_probs)
print("Ensembled test probabilities shape:", test_probs_ensemble.shape)


Ensembled test probabilities shape: (500, 5)


## 7. Build ranked top-3 predictions & submission file

In [9]:
def probs_to_top3_strings(probs):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    return [" ".join(OPTIONS[i] for i in row) for row in top3]


final_predictions = probs_to_top3_strings(test_probs_ensemble)

submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": final_predictions,
})

display(submission.head(10))


,ID,Prediction
0,1,A C E
1,2,B E D
2,3,B E D
3,4,E C A
4,5,C D A
5,6,D C A
6,7,E D A
7,8,B E C
8,9,C D E
9,10,B E D


## 8. Validate the submission format before writing it

Checks: same number of rows as `sample_submission.csv`, matching IDs, every `Prediction` cell has exactly
3 space-separated labels, all labels drawn from A–E, and no duplicate labels within a row.

In [10]:
assert submission.shape[0] == sample_submission.shape[0], "Row count mismatch with sample_submission.csv"
assert set(submission["ID"]) == set(sample_submission["ID"]), "ID mismatch with sample_submission.csv"
assert list(submission.columns) == ["ID", "Prediction"], "Column names/order must be ID,Prediction"

for pred in submission["Prediction"]:
    labels = pred.split(" ")
    assert len(labels) == 3, f"Expected exactly 3 labels, got: {pred}"
    assert len(set(labels)) == 3, f"Duplicate labels in prediction: {pred}"
    assert all(l in OPTIONS for l in labels), f"Invalid label in prediction: {pred}"

print("Submission format OK — matches sample_submission.csv structure.")


Submission format OK — matches sample_submission.csv structure.


In [11]:
submission = submission.sort_values("ID").reset_index(drop=True)
submission.to_csv("/kaggle/working/submission.csv", index=False)
print("submission.csv saved")
print("\nFinal submission shape:", submission.shape)
display(submission.head(10))


submission.csv saved

Final submission shape: (500, 2)


,ID,Prediction
0,1,A C E
1,2,B E D
2,3,B E D
3,4,E C A
4,5,C D A
5,6,D C A
6,7,E D A
7,8,B E C
8,9,C D E
9,10,B E D


In [12]:
assert os.path.exists("/kaggle/working/submission.csv"), "submission.csv not found!"
print("READY FOR KAGGLE SUBMISSION")
print("/kaggle/working/submission.csv")


READY FOR KAGGLE SUBMISSION
/kaggle/working/submission.csv


## Notes on pushing the score further

- **`microsoft/deberta-v3-large`** as `MODEL_NAME` is usually the single biggest lever on a text-classification/MCQ
  task like this — try it if your GPU time budget allows (increase `GRAD_ACCUM` or lower `TRAIN_BS` if you hit OOM).
- **`N_FOLDS`**: 5 is a good default for 2000 training rows; going to 8–10 folds trades more compute for a
  lower-variance ensemble.
- Because the model is now a genuine 5-way ranker (softmax over all options for the same question) rather than
  5 independent binary classifiers, its probabilities are directly comparable within a question — which is what
  MAP@3 needs.
- You can sanity-check calibration by inspecting `overall_oof_map3` printed above; if it's noticeably higher than
  what you see on the public leaderboard, the two most likely causes are (a) leaderboard test distribution shift or
  (b) too few folds relative to dataset size — try more folds.
